# TerraWatch Milestone 4 — Full-Data Change Detector Training, Ablations & Transfer

This notebook reproduces the full-data LEVIR-MCI change-detection study (**6,444 train / 1,242 val / 1,858 held-out test pairs**) using the repository's production modules:
- `src/models/detector_architectures.py` (unified 3-class factory: `bit_resnet18`, `ef_unet`, `fc_siam_diff`, `siam_r18_unet`, `changeformer_b2`)
- `scripts/train_detector_full.py` (AMP BF16/FP16 training, Focal + Soft Dice loss, exact resume & SHA-256 receipts)
- `scripts/eval_robustness_transfer.py` (held-out Test evaluation, 10-condition corruption ladder, Otsu/Always-BG controls, and zero-shot SECOND-CC transfer)

By default (`RUN_TRAINING = False`), the notebook replays and visualizes all **24 completed full-data detector runs** from `docs/milestone4/evidence/full_data_m4/`. Set `RUN_TRAINING = True` on a CUDA GPU (RTX 4060, T4, or L4) to train or re-evaluate checkpoints from scratch.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Auto-detect repository root (works locally and in Google Colab)
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/content/TerraWatch")]
REPO = next((p for p in CANDIDATES if (p / "scripts/train_detector_full.py").is_file()), None)
if REPO is None:
    REPO = Path("/content/TerraWatch")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/shoaib2000857/Group-7-DS-and-AI-Lab-Project.git", str(REPO)],
        check=True,
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

EVIDENCE_DIR = REPO / "docs/milestone4/evidence/full_data_m4"
CFG = json.loads((REPO / "configs/detector_training.json").read_text(encoding="utf-8"))
print("Repo root:", REPO)
print("Selected deployment detector:", CFG["selected_run_id"], "| arch:", CFG["architecture"], "| SHA-256:", CFG["selected_checkpoint_sha256"][:16])

## 1. Replay Full-Data 24-Run Detector Leaderboard & 3-Seed Stability

In [ ]:
agg = json.loads((EVIDENCE_DIR / "aggregated_summary.json").read_text(encoding="utf-8"))
rows = agg["detector_ranking"]
print(f"Total full-data detector runs logged: {len(rows)}\n")
print(f"{'Run ID':<24} {'Arch':<15} {'LR':<7} {'Chg mIoU':>9} {'Road':>7} {'Bldg':>7} {'Bin F1':>8} {'FA@50px':>8}")
print("-" * 92)
for r in rows:
    print(
        f"{r['run_id']:<24} {r['arch']:<15} {r['head_lr']:<7} "
        f"{r['val_change_miou']:>8.2f}% {r['val_road_iou']:>6.2f}% {r['val_building_iou']:>6.2f}% "
        f"{r['val_binary_f1']:>7.2f}% {r['fa_50px_pct']:>7.2f}%"
    )

print("\n3-Seed Stability (seeds 42, 0, 2026):")
for name, stats in agg["detector_multiseed"].items():
    print(
        f"  {name:<20} -> Val Change mIoU = {stats['val_change_miou_mean']:.2f}% ± {stats['val_change_miou_std']:.2f}% "
        f"| Val Bin F1 = {stats['val_binary_f1_mean']:.2f}% ± {stats['val_binary_f1_std']:.2f}%"
    )

## 2. Held-Out LEVIR-MCI Test (1,858 Pairs), Corruption Robustness & SECOND-CC Transfer

In [ ]:
bit_rob = json.loads((EVIDENCE_DIR / "robustness_and_transfer_bit.json").read_text(encoding="utf-8"))
siam_rob = json.loads((EVIDENCE_DIR / "robustness_and_transfer_siam.json").read_text(encoding="utf-8"))

print("=== Held-Out LEVIR-MCI Test (1,858 pairs) ===")
for label, d in [("BIT-ResNet18 (D03, Selected)", bit_rob["levir_mci_test"]), ("Siam-R18-UNet (A03)", siam_rob["levir_mci_test"])]:
    print(f"  {label:<30}: Change mIoU={d['change_miou']*100:.2f}%, Road={d['road_iou']*100:.2f}%, Bldg={d['building_iou']*100:.2f}%, Bin F1={d['binary_f1']*100:.2f}%, FA@50px={d['unchanged_fa_rates']['50']*100:.2f}%")

print("\n=== Key Corruption Conditions (1,242 Val Pairs — Change mIoU %) ===")
print(f"{'Condition':<22} {'BIT (D03)':>10} {'Siam (A03)':>11} {'BIT - Siam':>11}")
for cond in ["clean", "jpeg_q30", "jpeg_q15", "blur_r1.0", "blur_r2.0", "bright_dark", "bright_bright", "noise_s15", "rot_2deg", "shift_4px"]:
    b = bit_rob["levir_mci_val_corruptions"][cond]["change_miou"] * 100
    s = siam_rob["levir_mci_val_corruptions"][cond]["change_miou"] * 100
    print(f"{cond:<22} {b:>9.2f}% {s:>10.2f}% {b-s:>+10.2f}%")

print("\n=== Zero-Shot Cross-Dataset Transfer on SECOND-CC ===")
for split in ("val", "test"):
    bt = bit_rob["second_cc_transfer"][split]
    st = siam_rob["second_cc_transfer"][split]
    print(
        f"  SECOND-CC {split:<4} | BIT Any-F1={bt['binary_any_change']['f1']*100:.2f}%, Bldg-F1={bt['building_proxy']['f1']*100:.2f}% "
        f"| Siam Any-F1={st['binary_any_change']['f1']*100:.2f}%, Bldg-F1={st['building_proxy']['f1']*100:.2f}%"
    )

## 3. Train Selected Detector (`D03_bit_lr3e4`) or Any Architecture Baseline on GPU

Set `RUN_TRAINING = True` and point `DATA_DIR` to your extracted LEVIR-MCI directory (`train/`, `val/`, `test/`, and `manifests/`).

In [ ]:
RUN_TRAINING = False
DATA_DIR = Path(os.environ.get("TERRAWATCH_MCI_DIR", "data/processed/levir_mci"))
BIT_SRC = Path(os.environ.get("BIT_CD_DIR", "/content/BIT_CD"))
OUT_DIR = Path("outputs/detector_runs/D03_bit_lr3e4")

cmd = [
    sys.executable, "scripts/train_detector_full.py",
    "--run-id", "D03_bit_lr3e4",
    "--arch", "bit_resnet18",
    "--data-dir", str(DATA_DIR),
    "--bit-source-dir", str(BIT_SRC),
    "--out-dir", str(OUT_DIR),
    "--epochs", "30",
    "--batch-size", "16",
    "--grad-accum", "1",
    "--head-lr", "3e-4",
    "--backbone-lr", "3e-5",
    "--weight-decay", "1e-4",
    "--scheduler", "cosine",
    "--loss", "focal_dice",
    "--freeze-epochs", "2",
    "--aug", "geom",
    "--seed", "42",
    "--resume",
]
print("Detector command:\n ", " ".join(cmd))
if RUN_TRAINING:
    subprocess.run(cmd, check=True)

## 4. Run Held-Out Test, 10-Condition Robustness & SECOND-CC Transfer Evaluation

In [ ]:
RUN_EVAL = False
SECOND_CC_DIR = Path(os.environ.get("TERRAWATCH_SECOND_CC_DIR", "data/raw/second_cc"))

eval_cmd = [
    sys.executable, "scripts/eval_robustness_transfer.py",
    "--arch", "bit_resnet18",
    "--checkpoint", str(OUT_DIR / "best.pt"),
    "--data-dir", str(DATA_DIR),
    "--second-cc-dir", str(SECOND_CC_DIR),
    "--bit-source-dir", str(BIT_SRC),
    "--out-json", "outputs/robustness_and_transfer_bit.json",
]
print("Robustness & transfer command:\n ", " ".join(eval_cmd))
if RUN_EVAL:
    subprocess.run(eval_cmd, check=True)